# 00 — Native data exploration and statistical intake audit

**Outcome:** understand the source before defining a contract or fitting a model.

This notebook answers:

1. What files, rows, columns, entities and time ranges are actually present?
2. What is the unit of observation and sampling cadence?
3. Which measurements are missing, degenerate, censored, heavy-tailed or
   structurally different?
4. Which relationships and time-series behaviours deserve modelling?
5. What truth exists, and how must it be isolated from model inputs?

`EDA_SECTOR=telecom` is the default. Set it to `petrobras_3w` to repeat the
same statistical intake process on **real WELL files**. Simulated and hand-drawn
3W files are counted in the exact inventory but excluded from the default
analysis sample.

## Statistical discipline used here

The notebook deliberately separates three evidence levels:

- **Exact metadata:** file count, Parquet row count, schema and file size.
- **Exact selected-series analysis:** cadence, gaps and frozen runs for a small
  number of complete entity histories.
- **Sample estimates:** distributions, correlations, missingness and prevalence.

Every sample-based table is labelled. The operational measurements are analysed
without ground truth. Labels are opened only in the final evaluation-only
section. No canonical `SPEC-CORE` or modelling output is created here.

## 1. Setup and reproducible controls

In [ ]:
import json
import math
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import (
    CORE_VERSION,
    discover_telecom,
    discover_threew,
    native_path,
    write_json,
)

EDA_SECTOR = os.getenv("EDA_SECTOR", "telecom")
assert EDA_SECTOR in {"telecom", "petrobras_3w"}

EDA_RUN_ID = os.getenv(
    "EDA_RUN_ID", f"{EDA_SECTOR}_native_eda_v1"
)
RANDOM_SEED = int(os.getenv("EDA_RANDOM_SEED", "42"))
SAMPLE_ROWS = int(os.getenv("EDA_SAMPLE_ROWS", "200000"))
SERIES_ENTITY_COUNT = int(os.getenv("EDA_SERIES_ENTITY_COUNT", "2"))
SERIES_DAYS = int(os.getenv("EDA_SERIES_DAYS", "7"))
SERIES_MAX_POINTS = int(os.getenv("EDA_SERIES_MAX_POINTS", "3000"))
THREEW_FILE_COUNT = int(os.getenv(
    "EDA_THREEW_FILE_COUNT", "20"
))
THREEW_ROWS_PER_FILE = int(os.getenv(
    "EDA_THREEW_ROWS_PER_FILE", "10000"
))
INCLUDE_TRUTH_EDA = os.getenv("EDA_INCLUDE_TRUTH", "1") == "1"

TELECOM_SOURCE = Path(os.getenv(
    "TELECOM_SOURCE_ROOT",
    str(
        DRIVE_ROOT / "Full dataset"
        if (DRIVE_ROOT / "Full dataset").exists()
        else DRIVE_ROOT
    ),
))
THREEW_SOURCE = Path(os.getenv(
    "THREEW_SOURCE_ROOT",
    str(
        DRIVE_ROOT / "sources" / "petrobras_3w" / "2.0.0"
        / "raw" / "3w_dataset_2.0.0"
    ),
))
SOURCE_ROOT = (
    TELECOM_SOURCE if EDA_SECTOR == "telecom" else THREEW_SOURCE
)
OUTPUT = (
    DRIVE_ROOT / "outputs" / "exploration" / EDA_SECTOR / EDA_RUN_ID
)
FIGURES = OUTPUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
rng = np.random.default_rng(RANDOM_SEED)

display(pd.Series({
    "sector": EDA_SECTOR,
    "source": str(SOURCE_ROOT),
    "output": str(OUTPUT),
    "distribution_sample_rows": SAMPLE_ROWS,
    "series_entity_count": SERIES_ENTITY_COUNT,
    "random_seed": RANDOM_SEED,
    "truth_eda_enabled": INCLUDE_TRUTH_EDA,
}, name="value").to_frame())

In [ ]:
def parquet_shape(path):
    parquet = pq.ParquetFile(path)
    return parquet.metadata.num_rows, len(parquet.schema_arrow.names)


def csv_shape(path):
    columns = len(pd.read_csv(path, nrows=0).columns)
    with Path(path).open("r", encoding="utf-8", errors="replace") as handle:
        rows = max(0, sum(1 for _ in handle) - 1)
    return rows, columns


def timestamp_bounds(parquet, field):
    if field not in parquet.schema.names:
        return pd.NaT, pd.NaT
    column_index = parquet.schema.names.index(field)
    starts, ends = [], []
    for index in range(parquet.num_row_groups):
        statistics = parquet.metadata.row_group(index).column(
            column_index
        ).statistics
        if statistics is not None and statistics.has_min_max:
            starts.append(pd.to_datetime(statistics.min, utc=True))
            ends.append(pd.to_datetime(statistics.max, utc=True))
    return (
        min(starts) if starts else pd.NaT,
        max(ends) if ends else pd.NaT,
    )


def save_figure(figure, filename):
    figure.tight_layout()
    figure.savefig(FIGURES / filename, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(figure)


def threew_source_kind(path):
    if path.stem.startswith("WELL-"):
        return "real"
    if path.stem.startswith("SIMULATED_"):
        return "simulated"
    if path.stem.startswith("DRAWN_"):
        return "hand_drawn"
    return "other"

## 2. Exact source inventory

This is not a sample. Parquet row counts come from file metadata. For 3W the
cell opens the metadata of all 2,228 instance files; on Google Drive this can be
the slowest EDA step, but it establishes the true scale and composition.

In [ ]:
if EDA_SECTOR == "telecom":
    discovery = discover_telecom(SOURCE_ROOT)
    assert discovery["core_ready"], discovery

    file_specs = [
        ("reference_dataset.parquet", "operational_panel", False, True),
        ("topology.csv", "operational_topology", False, True),
        ("entity_service_windows.csv", "operational_validity", False, True),
        ("engineering_events.csv", "operational_context", False, False),
        ("fault_entity_intervals.csv", "evaluation_truth", True, False),
        ("gt_fault_groups.csv", "evaluation_truth", True, False),
        ("gt_fault_registry.csv", "evaluation_truth", True, False),
        ("gt_benign_anomalies.csv", "evaluation_truth", True, False),
        ("gt_collection_gaps.parquet", "evaluation_truth", True, False),
        ("tickets.csv", "evaluation_truth", True, False),
    ]
    inventory_rows = []
    for filename, role, evaluation, required in file_specs:
        path = native_path(
            SOURCE_ROOT,
            filename,
            evaluation=evaluation,
            required=False,
        )
        if path is None:
            inventory_rows.append({
                "file": filename,
                "role": role,
                "present": False,
                "rows": None,
                "columns": None,
                "size_mb": None,
                "path": None,
            })
            continue
        rows, columns = (
            parquet_shape(path)
            if path.suffix == ".parquet"
            else csv_shape(path)
        )
        inventory_rows.append({
            "file": filename,
            "role": role,
            "present": True,
            "rows": rows,
            "columns": columns,
            "size_mb": path.stat().st_size / 1024 ** 2,
            "path": str(path),
        })
    source_inventory = pd.DataFrame(inventory_rows)
    panel_path = native_path(
        SOURCE_ROOT, "reference_dataset.parquet"
    )
    panel_parquet = pq.ParquetFile(panel_path)
    exact_row_count = panel_parquet.metadata.num_rows
    exact_column_count = len(panel_parquet.schema_arrow.names)
    exact_file_count = int(source_inventory["present"].sum())
    exact_start, exact_end = timestamp_bounds(
        panel_parquet, "timestamp_utc"
    )
    file_inventory = pd.DataFrame()
    display(source_inventory)
else:
    discovery = discover_threew(SOURCE_ROOT)
    assert discovery["ready"], discovery
    paths = sorted(
        path
        for event_code in range(10)
        for path in (SOURCE_ROOT / str(event_code)).glob("*.parquet")
    )
    inventory_rows = []
    for position, path in enumerate(paths, start=1):
        parquet = pq.ParquetFile(path)
        start, end = timestamp_bounds(parquet, "timestamp")
        kind = threew_source_kind(path)
        entity_id = (
            path.stem.split("_", 1)[0] if kind == "real"
            else path.stem
        )
        inventory_rows.append({
            "relative_path": str(path.relative_to(SOURCE_ROOT)),
            "source_kind": kind,
            "entity_id": entity_id,
            "rows": parquet.metadata.num_rows,
            "columns": len(parquet.schema_arrow.names),
            "start_ts": start,
            "end_ts": end,
            "size_mb": path.stat().st_size / 1024 ** 2,
            "path": str(path),
        })
        if position % 250 == 0:
            print(f"Scanned metadata for {position:,}/{len(paths):,} files")
    file_inventory = pd.DataFrame(inventory_rows)
    source_inventory = (
        file_inventory.groupby("source_kind", as_index=False)
        .agg(
            files=("relative_path", "size"),
            rows=("rows", "sum"),
            entities=("entity_id", "nunique"),
            size_mb=("size_mb", "sum"),
        )
    )
    exact_row_count = int(file_inventory["rows"].sum())
    exact_column_count = int(file_inventory["columns"].max())
    exact_file_count = len(file_inventory)
    exact_start = file_inventory["start_ts"].min()
    exact_end = file_inventory["end_ts"].max()
    panel_path = None
    panel_parquet = None
    display(source_inventory)

source_inventory.to_csv(OUTPUT / "source_inventory.csv", index=False)
display(pd.Series({
    "exact_files": exact_file_count,
    "exact_rows": exact_row_count,
    "maximum_columns_per_record": exact_column_count,
    "metadata_time_start": exact_start,
    "metadata_time_end": exact_end,
}, name="exact_value").to_frame())

## 3. Schema, grain and leakage boundary

The first modelling question is the grain: one telecom row is one ONT poll,
while one 3W row is one timestamp inside one well-event file. Embedded labels
are catalogued here but excluded from `operational_sample`.

In [ ]:
TELECOM_METRICS = [
    "rx_power_dbm", "tx_power_dbm", "temperature_c",
    "bias_current_ma", "voltage_v", "ber", "fec_count",
    "crc_errors", "uptime_s", "reboot_count", "throughput_mbps",
]
TELECOM_CONTEXT = [
    "olt_id", "pon_port", "splitter_l1", "splitter_l2",
    "geo_cluster", "device_model", "vendor", "enclosure",
    "firmware_version", "distance_m", "distance_bucket",
    "splitter_ratio", "l2_splitter_capacity", "fibre_age_yr",
    "service_impact_weight", "customer_priority_weight",
]
TELECOM_TRUTH_SAMPLE = [
    "gt_state", "gt_fault_id", "gt_fault_type", "gt_onset_ts",
    "gt_impact_ts", "gt_repair_ts", "gt_left_censored",
    "gt_shared_fault", "gt_active_fault_count",
]

if EDA_SECTOR == "telecom":
    schema = pd.DataFrame([
        {"field": field.name, "physical_type": str(field.type)}
        for field in panel_parquet.schema_arrow
    ])
    all_fields = set(schema["field"])
    metric_fields = [
        field for field in TELECOM_METRICS if field in all_fields
    ]
    context_fields = [
        field for field in TELECOM_CONTEXT if field in all_fields
    ]
    truth_fields = sorted(
        field for field in all_fields if field.startswith("gt_")
    )
    truth_sample_fields = [
        field for field in TELECOM_TRUTH_SAMPLE if field in all_fields
    ]
    timestamp_field = "timestamp_utc"
    entity_field = "ont_id"
    grain = "one ONT poll at one timestamp"
else:
    first_parquet = pq.ParquetFile(file_inventory.iloc[0]["path"])
    schema = pd.DataFrame([
        {"field": field.name, "physical_type": str(field.type)}
        for field in first_parquet.schema_arrow
    ])
    excluded = {"timestamp", "class", "state"}
    metric_fields = [
        field for field in schema["field"] if field not in excluded
    ]
    context_fields = []
    truth_fields = ["event_directory", "class", "state"]
    truth_sample_fields = ["class", "state"]
    timestamp_field = "timestamp"
    entity_field = "entity_id"
    grain = "one timestamp within one well-event instance file"

schema["role"] = np.select(
    [
        schema["field"].isin(metric_fields),
        schema["field"].isin(truth_fields),
        schema["field"].eq(timestamp_field),
    ],
    ["operational_measurement", "evaluation_only", "timestamp"],
    default="context_or_identifier",
)
display(pd.Series({
    "grain": grain,
    "operational_metric_count": len(metric_fields),
    "truth_fields": ", ".join(truth_fields),
}, name="value").to_frame())
display(schema)
schema.to_csv(OUTPUT / "native_schema.csv", index=False)

## 4. Reproducible bounded operational sample

Telecom is sampled proportionally from every Parquet row group. 3W draws real
`WELL-` files uniformly, then caps rows per selected file so unusually long
instances do not dominate. No label value or event directory is used to choose
or analyse the operational sample. The manifest records the exact design.

In [ ]:
if EDA_SECTOR == "telecom":
    read_columns = list(dict.fromkeys([
        timestamp_field,
        entity_field,
        *metric_fields,
        *context_fields,
    ]))
    row_group_rows = np.array([
        panel_parquet.metadata.row_group(index).num_rows
        for index in range(panel_parquet.num_row_groups)
    ])
    allocations = np.maximum(
        1,
        np.round(
            SAMPLE_ROWS * row_group_rows / row_group_rows.sum()
        ).astype(int),
    )
    sample_parts = []
    manifest_rows = []
    for row_group, requested in enumerate(allocations):
        frame = panel_parquet.read_row_group(
            row_group, columns=read_columns
        ).to_pandas()
        take = min(len(frame), int(requested))
        selected = frame.sample(
            n=take, random_state=RANDOM_SEED + row_group
        )
        sample_parts.append(selected)
        manifest_rows.append({
            "row_group": row_group,
            "source_rows": len(frame),
            "sample_rows": take,
        })
    analysis_sample = pd.concat(sample_parts, ignore_index=True)
    if len(analysis_sample) > SAMPLE_ROWS:
        analysis_sample = analysis_sample.sample(
            n=SAMPLE_ROWS, random_state=RANDOM_SEED
        ).reset_index(drop=True)
    sample_manifest = pd.DataFrame(manifest_rows)
    sample_design = "proportional random sample from every row group"
else:
    real_inventory = file_inventory.loc[
        file_inventory["source_kind"].eq("real")
    ]
    take_files = min(THREEW_FILE_COUNT, len(real_inventory))
    chosen = rng.choice(
        real_inventory.index.to_numpy(),
        size=take_files,
        replace=False,
    )
    selected_files = (
        real_inventory.loc[chosen]
        .sort_values("relative_path")
        .reset_index(drop=True)
    )
    sample_parts = []
    manifest_rows = []
    for position, record in selected_files.iterrows():
        frame = pd.read_parquet(
            record["path"], columns=metric_fields
        )
        if timestamp_field not in frame.columns:
            frame = frame.reset_index()
        take = min(len(frame), THREEW_ROWS_PER_FILE)
        selected = frame.sample(
            n=take, random_state=RANDOM_SEED + position
        ).copy()
        selected[entity_field] = record["entity_id"]
        selected["source_file"] = record["relative_path"]
        selected["source_kind"] = record["source_kind"]
        sample_parts.append(selected)
        manifest_rows.append({
            "relative_path": record["relative_path"],
            "entity_id": record["entity_id"],
            "source_kind": record["source_kind"],
            "source_rows": len(frame),
            "sample_rows": take,
        })
    analysis_sample = pd.concat(sample_parts, ignore_index=True)
    sample_manifest = pd.DataFrame(manifest_rows)
    sample_design = (
        "uniform random sample of real-WELL files; "
        "equal bounded rows per selected file"
    )

analysis_sample[timestamp_field] = pd.to_datetime(
    analysis_sample[timestamp_field], utc=True, errors="coerce"
)
operational_columns = list(dict.fromkeys([
    timestamp_field, entity_field, *metric_fields, *context_fields
]))
operational_sample = analysis_sample[operational_columns].copy()

sample_key = (
    [entity_field, timestamp_field]
    if EDA_SECTOR == "telecom"
    else ["source_file", timestamp_field]
)
sample_duplicate_keys = int(
    analysis_sample.duplicated(sample_key).sum()
)

sample_manifest.to_csv(OUTPUT / "sample_manifest.csv", index=False)
display(pd.Series({
    "sample_design": sample_design,
    "sample_rows": len(operational_sample),
    "sample_entities": operational_sample[entity_field].nunique(),
    "sample_time_start": operational_sample[timestamp_field].min(),
    "sample_time_end": operational_sample[timestamp_field].max(),
    "duplicate_grain_keys_in_sample": sample_duplicate_keys,
}, name="sample_value").to_frame())
display(sample_manifest.head(25))

## 5. Descriptive statistics and distribution diagnostics

In [ ]:
def numeric_summary(frame, fields):
    rows = []
    for field in fields:
        values = pd.to_numeric(frame[field], errors="coerce")
        finite_mask = np.isfinite(values)
        finite = values.loc[finite_mask]
        quantiles = finite.quantile(
            [0.01, 0.25, 0.50, 0.75, 0.99]
        ) if len(finite) else pd.Series(dtype=float)
        rows.append({
            "field": field,
            "sample_rows": len(values),
            "valid_count": int(len(finite)),
            "missing_fraction": float(values.isna().mean()),
            "infinite_count": int((values.notna() & ~finite_mask).sum()),
            "unique_values": int(finite.nunique()),
            "zero_fraction_of_valid": (
                float(finite.eq(0).mean()) if len(finite) else None
            ),
            "mean": float(finite.mean()) if len(finite) else None,
            "std": float(finite.std()) if len(finite) else None,
            "min": float(finite.min()) if len(finite) else None,
            "p01": float(quantiles.get(0.01, np.nan)),
            "p25": float(quantiles.get(0.25, np.nan)),
            "median": float(quantiles.get(0.50, np.nan)),
            "p75": float(quantiles.get(0.75, np.nan)),
            "p99": float(quantiles.get(0.99, np.nan)),
            "max": float(finite.max()) if len(finite) else None,
            "skew": float(finite.skew()) if len(finite) > 2 else None,
            "sample_min_fraction": (
                float(finite.eq(finite.min()).mean())
                if len(finite) else None
            ),
            "sample_max_fraction": (
                float(finite.eq(finite.max()).mean())
                if len(finite) else None
            ),
        })
    return pd.DataFrame(rows)


summary = numeric_summary(operational_sample, metric_fields)
summary.to_csv(OUTPUT / "numeric_summary_sample.csv", index=False)
display(summary)

In [ ]:
quality_rows = []
for row in summary.itertuples(index=False):
    checks = [
        (
            row.missing_fraction >= 0.05,
            "material_missingness",
            row.missing_fraction,
        ),
        (
            row.zero_fraction_of_valid is not None
            and row.zero_fraction_of_valid >= 0.50,
            "zero_inflated",
            row.zero_fraction_of_valid,
        ),
        (
            row.unique_values <= 2,
            "low_cardinality_or_state",
            row.unique_values,
        ),
        (
            row.sample_max_fraction is not None
            and row.sample_max_fraction >= 0.10,
            "upper_boundary_concentration",
            row.sample_max_fraction,
        ),
        (
            row.skew is not None
            and math.isfinite(row.skew)
            and abs(row.skew) >= 5,
            "extreme_skew",
            row.skew,
        ),
        (
            row.infinite_count > 0,
            "infinite_values",
            row.infinite_count,
        ),
    ]
    for triggered, issue, evidence in checks:
        if triggered:
            quality_rows.append({
                "field": row.field,
                "issue_to_investigate": issue,
                "sample_evidence": evidence,
            })

quality_flags = pd.DataFrame(
    quality_rows,
    columns=["field", "issue_to_investigate", "sample_evidence"],
)
quality_flags.to_csv(OUTPUT / "quality_flags_sample.csv", index=False)
display(quality_flags)

In [ ]:
figure, axis = plt.subplots(figsize=(10, max(4, len(summary) * 0.35)))
missing = summary.sort_values("missing_fraction")
axis.barh(missing["field"], missing["missing_fraction"], color="#4C78A8")
axis.set_xlim(0, max(0.01, missing["missing_fraction"].max() * 1.1))
axis.set_xlabel("Missing fraction in analysis sample")
axis.set_title(f"{EDA_SECTOR}: operational measurement missingness")
axis.grid(axis="x", alpha=0.25)
save_figure(figure, "01_missingness.png")

### 5.1 Missingness heterogeneity by entity

A low global missing fraction can conceal an entity whose channel is almost
entirely absent. These are sample estimates per entity; complete-series checks
later provide exact values for the selected examples.

In [ ]:
entity_missingness_wide = (
    operational_sample.groupby(entity_field, observed=True)[metric_fields]
    .agg(lambda values: values.isna().mean())
)
entity_missingness_summary = pd.DataFrame([
    {
        "field": field,
        "sample_entity_median_missing_fraction": float(
            entity_missingness_wide[field].median()
        ),
        "sample_entity_p95_missing_fraction": float(
            entity_missingness_wide[field].quantile(0.95)
        ),
        "sample_entity_max_missing_fraction": float(
            entity_missingness_wide[field].max()
        ),
        "sample_entities_fully_missing": int(
            entity_missingness_wide[field].eq(1).sum()
        ),
    }
    for field in metric_fields
])
entity_missingness_summary.to_csv(
    OUTPUT / "entity_missingness_summary_sample.csv", index=False
)
display(entity_missingness_summary)

figure, axis = plt.subplots(
    figsize=(10, max(4, len(metric_fields) * 0.42))
)
axis.boxplot(
    [
        entity_missingness_wide[field].dropna().to_numpy()
        for field in metric_fields
    ],
    vert=False,
    labels=metric_fields,
    showfliers=False,
)
axis.set_xlabel("Entity-level missing fraction in analysis sample")
axis.set_title(f"{EDA_SECTOR}: missingness heterogeneity by entity")
axis.grid(axis="x", alpha=0.25)
save_figure(figure, "01b_entity_missingness.png")

In [ ]:
requested_plot_metrics = [
    item.strip()
    for item in os.getenv("EDA_PLOT_METRICS", "").split(",")
    if item.strip()
]
default_plot_metrics = (
    TELECOM_METRICS
    if EDA_SECTOR == "telecom"
    else [
        "P-ANULAR", "P-MON-CKP", "P-TPT", "QGL",
        "T-TPT", "ABER-CKP", "ESTADO-DHSV", "ESTADO-M1",
    ]
)
plot_metrics = [
    field
    for field in (requested_plot_metrics or default_plot_metrics)
    if field in metric_fields
][:12]

columns = 3
rows = math.ceil(len(plot_metrics) / columns)
figure, axes = plt.subplots(
    rows, columns, figsize=(15, 3.6 * rows), squeeze=False
)
for axis, field in zip(axes.flat, plot_metrics):
    values = pd.to_numeric(
        operational_sample[field], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan).dropna()
    if values.nunique() <= 1:
        axis.text(0.5, 0.5, "constant / unavailable", ha="center")
        axis.set_title(field)
        continue
    lower, upper = values.quantile([0.005, 0.995])
    bulk = values.clip(lower=lower, upper=upper)
    axis.hist(bulk, bins=50, color="#4C78A8", alpha=0.85)
    axis.set_title(field)
    axis.set_xlabel("value, winsorised to sample 0.5%–99.5%")
    axis.set_ylabel("sample rows")
    axis.grid(alpha=0.2)
for axis in axes.flat[len(plot_metrics):]:
    axis.axis("off")
figure.suptitle(
    f"{EDA_SECTOR}: sample distributions (bulk view)",
    y=1.01,
)
save_figure(figure, "02_distributions.png")

The histograms intentionally winsorise only the visual display. The summary
table preserves the true sample minima and maxima. This keeps rare extremes
from hiding the bulk distribution without silently deleting them.

## 6. Dependence structure

In [ ]:
correlation_fields = [
    field for field in metric_fields
    if operational_sample[field].notna().sum() >= 50
    and operational_sample[field].nunique(dropna=True) > 1
]
correlation_sample = operational_sample[correlation_fields]
if len(correlation_sample) > 50000:
    correlation_sample = correlation_sample.sample(
        n=50000, random_state=RANDOM_SEED
    )
correlation = correlation_sample.corr(method="spearman")
correlation.to_csv(OUTPUT / "spearman_correlation_sample.csv")

figure, axis = plt.subplots(
    figsize=(max(8, len(correlation) * 0.65), max(7, len(correlation) * 0.6))
)
image = axis.imshow(correlation, vmin=-1, vmax=1, cmap="coolwarm")
axis.set_xticks(range(len(correlation)))
axis.set_yticks(range(len(correlation)))
axis.set_xticklabels(correlation.columns, rotation=90)
axis.set_yticklabels(correlation.index)
axis.set_title(
    f"{EDA_SECTOR}: sample Spearman dependence\n"
    "(descriptive association, not causal evidence)"
)
figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
save_figure(figure, "03_spearman_correlation.png")

Correlation is exploratory, not a feature-selection verdict. Shared seasonality,
event regime, topology and sampling artefacts can all create association. For
3W, pooled correlations also mix event types; later work should compare
within-well and within-regime dependence.

## 7. Complete time-series examples and temporal quality

In [ ]:
if EDA_SECTOR == "telecom":
    requested_entities = [
        item.strip()
        for item in os.getenv("EDA_SERIES_ENTITY_IDS", "").split(",")
        if item.strip()
    ]
    series_entities = (
        requested_entities
        or sorted(
            operational_sample[entity_field].dropna().astype(str).unique()
        )[:SERIES_ENTITY_COUNT]
    )
    series_metrics = [
        field for field in [
            "rx_power_dbm", "temperature_c", "ber",
            "fec_count", "crc_errors", "throughput_mbps",
        ] if field in metric_fields
    ]
    series_parts = []
    read_columns = [
        timestamp_field, entity_field, *series_metrics
    ]
    for batch in panel_parquet.iter_batches(
        batch_size=100000, columns=read_columns
    ):
        frame = batch.to_pandas()
        selected = frame.loc[
            frame[entity_field].astype(str).isin(series_entities)
        ].copy()
        if len(selected):
            series_parts.append(selected)
    series_data = pd.concat(series_parts, ignore_index=True)
    series_data["series_id"] = series_data[entity_field].astype(str)
else:
    chosen_records = [
        row
        for _, row in sample_manifest.head(
            max(1, SERIES_ENTITY_COUNT)
        ).iterrows()
    ]
    series_metrics = [
        field for field in [
            "P-ANULAR", "P-MON-CKP", "P-TPT", "QGL", "T-TPT"
        ] if field in metric_fields
    ]
    series_parts = []
    for record in chosen_records:
        path = SOURCE_ROOT / record["relative_path"]
        frame = pd.read_parquet(path, columns=series_metrics)
        if timestamp_field not in frame.columns:
            frame = frame.reset_index()
        frame[entity_field] = record["entity_id"]
        frame["series_id"] = record["relative_path"]
        series_parts.append(frame)
    series_data = pd.concat(series_parts, ignore_index=True)

series_data[timestamp_field] = pd.to_datetime(
    series_data[timestamp_field], utc=True, errors="coerce"
)
series_data = series_data.sort_values(
    ["series_id", timestamp_field]
).reset_index(drop=True)
print(
    f"Loaded {len(series_data):,} complete-series rows across "
    f"{series_data['series_id'].nunique()} selected series."
)

In [ ]:
temporal_rows = []
series_metric_rows = []
for series_id, frame in series_data.groupby("series_id", sort=True):
    timestamps = frame[timestamp_field].dropna().sort_values()
    duplicate_timestamps = int(timestamps.duplicated().sum())
    differences = timestamps.drop_duplicates().diff().dropna()
    positive = differences.loc[differences.gt(pd.Timedelta(0))]
    cadence = positive.median() if len(positive) else pd.NaT
    gap_threshold = (
        cadence * 1.5 if pd.notna(cadence) else pd.NaT
    )
    temporal_rows.append({
        "series_id": series_id,
        "rows": len(frame),
        "start_ts": timestamps.min(),
        "end_ts": timestamps.max(),
        "duplicate_timestamps": duplicate_timestamps,
        "median_cadence_seconds": (
            cadence.total_seconds() if pd.notna(cadence) else None
        ),
        "p99_gap_seconds": (
            positive.quantile(0.99).total_seconds()
            if len(positive) else None
        ),
        "gaps_over_1_5x_cadence": (
            int(positive.gt(gap_threshold).sum())
            if pd.notna(gap_threshold) else None
        ),
    })
    for field in series_metrics:
        values = pd.to_numeric(frame[field], errors="coerce")
        observed = values.dropna()
        series_metric_rows.append({
            "series_id": series_id,
            "field": field,
            "missing_fraction": float(values.isna().mean()),
            "unchanged_step_fraction": (
                float(observed.diff().eq(0).mean())
                if len(observed) > 1 else None
            ),
            "standard_deviation": (
                float(observed.std()) if len(observed) else None
            ),
        })

temporal_quality = pd.DataFrame(temporal_rows)
series_metric_quality = pd.DataFrame(series_metric_rows)
temporal_quality.to_csv(
    OUTPUT / "temporal_quality_selected_series.csv", index=False
)
series_metric_quality.to_csv(
    OUTPUT / "measurement_quality_selected_series.csv", index=False
)
display(temporal_quality)
display(
    series_metric_quality.sort_values(
        ["unchanged_step_fraction", "missing_fraction"],
        ascending=False,
    )
)

In [ ]:
series_ids = list(series_data["series_id"].drop_duplicates())
figure, axes = plt.subplots(
    len(series_ids),
    len(series_metrics),
    figsize=(4.2 * len(series_metrics), 3.0 * len(series_ids)),
    squeeze=False,
)
for row_index, series_id in enumerate(series_ids):
    frame = series_data.loc[
        series_data["series_id"].eq(series_id)
    ].sort_values(timestamp_field)
    if EDA_SECTOR == "telecom" and SERIES_DAYS > 0:
        cutoff = frame[timestamp_field].max() - pd.Timedelta(
            days=SERIES_DAYS
        )
        frame = frame.loc[frame[timestamp_field].ge(cutoff)]
    if len(frame) > SERIES_MAX_POINTS:
        positions = np.linspace(
            0, len(frame) - 1, SERIES_MAX_POINTS, dtype=int
        )
        frame = frame.iloc[positions]
    for column_index, field in enumerate(series_metrics):
        axis = axes[row_index, column_index]
        axis.plot(
            frame[timestamp_field],
            pd.to_numeric(frame[field], errors="coerce"),
            linewidth=0.8,
        )
        axis.set_title(f"{series_id}\n{field}", fontsize=9)
        axis.grid(alpha=0.2)
        axis.tick_params(axis="x", rotation=30)
figure.suptitle(
    f"{EDA_SECTOR}: selected complete time-series views",
    y=1.01,
)
save_figure(figure, "04_selected_time_series.png")

## 8. Context composition

Context imbalance can masquerade as anomaly structure. For telecom this table
shows vendor, firmware, topology and physical-context composition. 3W contains
little native non-label context, which is itself an important limitation.

In [ ]:
categorical_rows = []
for field in context_fields:
    values = operational_sample[field]
    counts = values.astype("string").value_counts(dropna=False)
    categorical_rows.append({
        "field": field,
        "sample_unique_values": int(values.nunique(dropna=True)),
        "sample_missing_fraction": float(values.isna().mean()),
        "most_common_value": str(counts.index[0]) if len(counts) else None,
        "most_common_fraction": (
            float(counts.iloc[0] / len(values)) if len(values) else None
        ),
    })
categorical_summary = pd.DataFrame(
    categorical_rows,
    columns=[
        "field", "sample_unique_values", "sample_missing_fraction",
        "most_common_value", "most_common_fraction",
    ],
)
categorical_summary.to_csv(
    OUTPUT / "categorical_summary_sample.csv", index=False
)
display(
    categorical_summary
    if len(categorical_summary)
    else pd.DataFrame({
        "finding": [
            "No non-label context fields are native to this source."
        ]
    })
)

## 9. Evaluation-only exploration — open the truth box

Everything above used operational fields only. This final section reopens the
same sampled rows with label columns and separately reports fixture composition.

Do not use prevalence, fault type, event directory, `class`, `state`, tickets,
or any `gt_*` field as a detector feature, threshold input or resampling key.
Set `EDA_INCLUDE_TRUTH=0` to skip this section completely.

In [ ]:
evaluation_rows = []
if INCLUDE_TRUTH_EDA:
    if EDA_SECTOR == "telecom":
        truth_parts = []
        truth_columns = list(dict.fromkeys([
            timestamp_field, entity_field, *truth_sample_fields
        ]))
        for record in sample_manifest.itertuples(index=False):
            frame = panel_parquet.read_row_group(
                int(record.row_group), columns=truth_columns
            ).to_pandas()
            selected = frame.sample(
                n=int(record.sample_rows),
                random_state=RANDOM_SEED + int(record.row_group),
            )
            truth_parts.append(selected)
        truth_analysis_sample = pd.concat(
            truth_parts, ignore_index=True
        )
        if len(truth_analysis_sample) > SAMPLE_ROWS:
            truth_analysis_sample = truth_analysis_sample.sample(
                n=SAMPLE_ROWS, random_state=RANDOM_SEED
            ).reset_index(drop=True)

        for field in ["gt_state", "gt_fault_type"]:
            if field not in truth_analysis_sample:
                continue
            counts = (
                truth_analysis_sample[field].astype("string")
                .value_counts(dropna=False)
            )
            for label, count in counts.items():
                evaluation_rows.append({
                    "truth_dimension": field,
                    "label": str(label),
                    "sample_rows": int(count),
                    "sample_fraction": float(
                        count / len(truth_analysis_sample)
                    ),
                })
        fault_active = (
            truth_analysis_sample["gt_fault_id"].notna()
            & truth_analysis_sample["gt_fault_id"].astype(str).ne("")
            if "gt_fault_id" in truth_analysis_sample
            else pd.Series(False, index=truth_analysis_sample.index)
        )
        display(pd.Series({
            "embedded_truth_columns_in_native_panel": len(truth_fields),
            "truth_columns_loaded_for_eval_eda": len(
                truth_sample_fields
            ),
            "sample_rows_with_fault_id_fraction": fault_active.mean(),
        }, name="evaluation_only_value").to_frame())
    else:
        evaluation_file_inventory = file_inventory.copy()
        evaluation_file_inventory["event_code"] = (
            evaluation_file_inventory["relative_path"].map(
                lambda value: int(Path(value).parts[0])
            )
        )
        exact_label_composition = (
            evaluation_file_inventory.groupby(
                ["event_code", "source_kind"], as_index=False
            )
            .agg(files=("relative_path", "size"), rows=("rows", "sum"))
        )
        display(exact_label_composition)

        truth_parts = []
        for position, record in sample_manifest.iterrows():
            path = SOURCE_ROOT / record["relative_path"]
            frame = pd.read_parquet(
                path, columns=truth_sample_fields
            )
            if timestamp_field not in frame.columns:
                frame = frame.reset_index()
            selected = frame.sample(
                n=int(record["sample_rows"]),
                random_state=RANDOM_SEED + position,
            ).copy()
            selected["source_file"] = record["relative_path"]
            selected["event_code"] = int(
                Path(record["relative_path"]).parts[0]
            )
            truth_parts.append(selected)
        truth_analysis_sample = pd.concat(
            truth_parts, ignore_index=True
        )
        if "class" in truth_analysis_sample:
            counts = truth_analysis_sample["class"].value_counts(
                dropna=False
            )
            for label, count in counts.items():
                evaluation_rows.append({
                    "truth_dimension": "class",
                    "label": str(label),
                    "sample_rows": int(count),
                    "sample_fraction": float(
                        count / len(truth_analysis_sample)
                    ),
                })
        evaluation_file_inventory.drop(columns="path").to_csv(
            OUTPUT / "threew_file_inventory_evaluation_only.csv",
            index=False,
        )
        display(pd.Series({
            "analysis_source_kind": "real WELL files only",
            "selected_real_files": len(sample_manifest),
            "simulated_files_in_exact_inventory": int(
                file_inventory["source_kind"].eq("simulated").sum()
            ),
            "hand_drawn_files_in_exact_inventory": int(
                file_inventory["source_kind"].eq("hand_drawn").sum()
            ),
        }, name="evaluation_only_value").to_frame())
else:
    print("Evaluation-only EDA skipped; no label values were read.")

evaluation_summary = pd.DataFrame(
    evaluation_rows,
    columns=[
        "truth_dimension", "label", "sample_rows", "sample_fraction"
    ],
)
evaluation_summary.to_csv(
    OUTPUT / "evaluation_only_summary.csv", index=False
)
display(evaluation_summary)

## 10. Statistical intake report and decision checklist

In [ ]:
exact_entity_count = None
if EDA_SECTOR == "telecom":
    topology_path = native_path(SOURCE_ROOT, "topology.csv")
    topology = pd.read_csv(topology_path)
    exact_entity_count = int(topology["ont_id"].astype(str).nunique())
else:
    exact_entity_count = int(
        file_inventory.loc[
            file_inventory["source_kind"].eq("real"), "entity_id"
        ].nunique()
    )

report = {
    "eda_version": "native_eda_v1",
    "contract_version_available_but_not_used_for_translation": CORE_VERSION,
    "sector": EDA_SECTOR,
    "source_root": str(SOURCE_ROOT),
    "output_root": str(OUTPUT),
    "grain": grain,
    "exact": {
        "file_count": exact_file_count,
        "row_count": exact_row_count,
        "maximum_columns_per_record": exact_column_count,
        "entity_count": exact_entity_count,
        "metadata_time_start": exact_start,
        "metadata_time_end": exact_end,
        "source_kind_composition": (
            {
                kind: {
                    "files": int(
                        file_inventory["source_kind"].eq(kind).sum()
                    ),
                    "rows": int(
                        file_inventory.loc[
                            file_inventory["source_kind"].eq(kind), "rows"
                        ].sum()
                    ),
                }
                for kind in sorted(
                    file_inventory["source_kind"].unique()
                )
            }
            if EDA_SECTOR == "petrobras_3w" else None
        ),
    },
    "sample": {
        "design": sample_design,
        "random_seed": RANDOM_SEED,
        "rows": len(operational_sample),
        "entities": int(operational_sample[entity_field].nunique()),
        "time_start": operational_sample[timestamp_field].min(),
        "time_end": operational_sample[timestamp_field].max(),
        "duplicate_grain_keys": sample_duplicate_keys,
    },
    "operational_metric_count": len(metric_fields),
    "truth_fields_identified": truth_fields,
    "quality_flag_count": len(quality_flags),
    "selected_complete_series": int(series_data["series_id"].nunique()),
    "truth_read_only_after_operational_eda": True,
    "truth_read_stage": (
        "evaluation_only_final_section"
        if INCLUDE_TRUTH_EDA else "not_read"
    ),
    "large_sample_data_was_not_copied": True,
    "runtime_versions": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "pyarrow": pyarrow.__version__,
    },
    "artifacts": [
        "source_inventory.csv",
        "native_schema.csv",
        "sample_manifest.csv",
        "numeric_summary_sample.csv",
        "entity_missingness_summary_sample.csv",
        "quality_flags_sample.csv",
        "spearman_correlation_sample.csv",
        "temporal_quality_selected_series.csv",
        "measurement_quality_selected_series.csv",
        "categorical_summary_sample.csv",
        "evaluation_only_summary.csv",
        "figures/",
    ],
}
write_json(OUTPUT / "eda_report.json", report, overwrite=True)

checklist = pd.DataFrame([
    {
        "question": "Is the physical source complete enough to start?",
        "evidence": str(discovery),
        "decision": "review",
    },
    {
        "question": "Is the row grain explicit?",
        "evidence": grain,
        "decision": "yes",
    },
    {
        "question": "Are exact facts separated from estimates?",
        "evidence": sample_design,
        "decision": "yes",
    },
    {
        "question": "Are truth fields identifiable and isolatable?",
        "evidence": ", ".join(truth_fields),
        "decision": "yes" if truth_fields else "review",
    },
    {
        "question": "Do data-quality findings require pack semantics?",
        "evidence": f"{len(quality_flags)} sample flags",
        "decision": "review before Notebook 01 or 02",
    },
])
checklist.to_csv(OUTPUT / "decision_checklist.csv", index=False)
display(checklist)
print("EDA artifacts written to:", OUTPUT)

## How to interpret this before continuing

Do not ask only whether a distribution “looks normal.” Ask:

- Is missingness random, entity-specific, event-specific or outside service?
- Does a spike at a boundary mean a physical limit, quantisation or censoring?
- Are zeros true zero opportunity, a reporting floor or missing encoded as zero?
- Are apparent correlations driven by time of day, topology, vendor or event type?
- Does one cadence apply to every entity and metric?
- Are frozen values sensor behaviour, operational state or data failure?
- Is the labelled problem an event, a persistent condition, or both?

After documenting those answers, run Notebook 01 for telecom or Notebook 02 for
3W. Notebook 00 informs the pack and modelling assumptions; it does not alter
the source or produce model-ready data.